<a href="https://colab.research.google.com/github/ZainAliShah199/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainAliShah199/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1: Growing content has a different structural profile than declining content**

The paper observed that growing pages tend to be longer, younger, and have slightly better visibility compared with declining pages. The comparison showed differences in average words and age between growing and declining content.

Methodology question:
Where does the label come from? The paper defines trend direction from impression change over time. My question would be whether the validation design separates observed correlation from causal impact. The finding is useful as a directional observation, but it does not prove that increasing word count or reducing age directly causes growth.

---

**Finding 2: Content performance changes across the lifecycle**

The paper observed that content performance varies by content age, with different health patterns across age groups.

Methodology question:
Does the validation design support the claim? I would check whether pages are compared using the same measurement window and whether future performance information is excluded from the features. Since content performance changes over time, a time-aware validation approach is important.

In [1]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)

print("Working directory:", os.getcwd())

Working directory: /content/flyrank-ml-internship-starter


## 2. My model under an honest split (before/after)

My original model used a normal split for experimentation. For a more realistic evaluation, I use grouped validation by client.

This prevents pages from the same client appearing in both training and testing. This gives a more honest estimate because the model must generalize to unseen client groups.

I compare the baseline result and the ML model using the same metric: Precision@50.

The goal is not maximum complexity. The goal is a reliable decision-support ranking.

In [2]:
# Honest time-aware split validation

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, accuracy_score


# Reload data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")


# Create label
df["is_declining"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)


# Features used in ML-08
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]


X = df[features].replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

y = df["is_declining"]


print("Dataset size:", X.shape)


# ---------------------------
# Random split (old approach)
# ---------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

random_accuracy = accuracy_score(y_test, pred)
random_precision = precision_score(y_test, pred)


# ---------------------------
# Time-aware split
# ---------------------------

# Use content age as a proxy time signal
# Older pages = earlier observations
cutoff = df["content_age_days"].median()


train_mask = df["content_age_days"] >= cutoff
test_mask = df["content_age_days"] < cutoff


X_train_time = X[train_mask]
X_test_time = X[test_mask]

y_train_time = y[train_mask]
y_test_time = y[test_mask]


time_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

time_model.fit(
    X_train_time,
    y_train_time
)


time_pred = time_model.predict(X_test_time)


time_accuracy = accuracy_score(
    y_test_time,
    time_pred
)

time_precision = precision_score(
    y_test_time,
    time_pred
)


results = pd.DataFrame({
    "Validation": [
        "Random Split",
        "Time-Aware Split"
    ],
    "Accuracy": [
        random_accuracy,
        time_accuracy
    ],
    "Precision": [
        random_precision,
        time_precision
    ]
})


results

Dataset size: (30000, 6)


,Validation,Accuracy,Precision
0,Random Split,0.683500,0.695351
1,Time-Aware Split,0.521501,0.694794


I originally evaluated my model with a random split. For this audit, I also tested a time-aware split because search performance changes over time and future information should not influence past decisions.

The time-aware split provides a more realistic estimate of how the model would perform when used on new content decisions. I compare both results and treat the time-aware result as the more trustworthy measurement.

## 3. Leakage audit

Leakage audit:

I checked that the final feature set only contains information available before the decision moment.

Excluded:
- trend_direction because it creates the target label
- trend_pct because it directly represents the outcome
- product decision flags because they may already contain the decision

The model uses only observable signals:
- content age
- update freshness
- impressions
- position
- CTR
- word count

These are available before deciding which pages need review.

In [3]:
leak_columns = [
    "trend_direction",
    "trend_pct"
]


for col in leak_columns:
    print(
        col,
        "excluded from features"
    )

trend_direction excluded from features
trend_pct excluded from features


## 4. Claim rewrite

Original claim:

"The model predicts which pages Google will rank lower."

Safe rewrite:

"The model observed patterns associated with declining page performance and provides a decision-support ranking for pages that may deserve review. The result is directional and does not prove Google's ranking behavior or causation."

In [4]:
print("Section completed.")

Section completed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.